# Cat Intent Classifier Training Notebook

This Google Colab notebook trains an intent classifier for the uploaded cat audio dataset. It does **not** translate literal cat language or exact words. It predicts one of three broad intent labels:

- `Food`
- `Isolation`
- `Brushing`

Dataset details expected by this notebook:

- Zip file name: `archive.zip`
- Audio files inside the zip: `dataset/dataset/*.wav`
- Filename convention: `C_NNNNN_BB_SS_OOOOO_RXX.wav`
- Example filenames: `B_ANI01_MC_FN_SIM01_101.wav`, `F_BAC01_MC_MN_SIM01_101.wav`, `I_BLE01_EU_FN_DEL01_2SEQ1.wav`
- `C` is the emission context / label: `B` brushing, `F` waiting for food, `I` isolation in an unfamiliar environment
- `NNNNN` is the cat ID, such as `ANI01`, `BAC01`, or `BLE01`
- `BB` is breed: `MC` Maine Coon, `EU` European Shorthair
- `SS` is sex: `FI` female intact, `FN` female neutered, `MI` male intact, `MN` male neutered
- `OOOOO` is owner ID
- `RXX` stores recording session plus vocalization counter, such as `101` -> session `1`, counter `01`

The split is grouped by `cat_id` to reduce leakage. Breed, sex, owner ID, and recording session are saved in the manifest only; they are not prediction labels.


In [ ]:
# Install dependencies.
!pip -q install tensorflow tensorflow_hub librosa scikit-learn numpy pandas


In [ ]:
# Mount Google Drive. The notebook can read archive.zip from Drive or from a manual upload.
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import json
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
import tensorflow_hub as hub
from google.colab import files
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

ARCHIVE_NAME = 'archive.zip'
LOCAL_ARCHIVE_PATH = Path('/content') / ARCHIVE_NAME
DRIVE_ARCHIVE_PATH = Path('/content/drive/MyDrive') / ARCHIVE_NAME
EXTRACT_DIR = Path('/content/cat_audio_archive')

TARGET_SR = 16_000
DURATION_SECONDS = 3.0
TARGET_SAMPLES = int(TARGET_SR * DURATION_SECONDS)
HOP_SAMPLES = TARGET_SAMPLES // 2
RANDOM_STATE = 42

# Augmentation is applied to training chunks only. Keep validation/test unmodified.
ENABLE_AUDIO_AUGMENTATION = True
AUGMENT_RANDOM_NOISE = True
AUGMENT_TIME_SHIFT = True
AUGMENT_PITCH_SHIFT = True
AUGMENT_NOISE_LEVEL = 0.008
AUGMENT_MAX_SHIFT_SECONDS = 0.18
AUGMENT_PITCH_STEPS = 0.5

LABEL_FROM_RAW = {
    'F': 'Food',
    'I': 'Isolation',
    'B': 'Brushing',
}
LABEL_ORDER = ['Food', 'Isolation', 'Brushing']
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABEL_ORDER)}
INDEX_TO_LABEL = {str(index): label for index, label in enumerate(LABEL_ORDER)}
NUM_CLASSES = len(LABEL_ORDER)

MODEL_VERSION = datetime.now().strftime('%Y%m%d_%H%M%S')
MODEL_OUTPUT_PATH = Path('cat_model.keras')
TIMESTAMPED_MODEL_OUTPUT_PATH = Path(f'cat_model_{MODEL_VERSION}.keras')
LABEL_MAP_OUTPUT_PATH = Path('label_map.json')
MANIFEST_OUTPUT_PATH = Path('manifest.csv')
HISTORY_PLOT_OUTPUT_PATH = Path(f'training_history_{MODEL_VERSION}.png')
MANIFEST_COLUMNS = [
    'filepath',
    'filename',
    'context_label',
    'human_label',
    'cat_id',
    'breed',
    'sex',
    'owner_id',
    'recording_session',
    'vocalization_counter',
    'sample_rate',
    'duration_seconds',
]

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(f'TensorFlow: {tf.__version__}')
print(f'Target sample rate: {TARGET_SR} Hz')
print(f'Window length: {DURATION_SECONDS} seconds ({TARGET_SAMPLES} samples)')


## Load `archive.zip`

This cell first checks for `/content/drive/MyDrive/archive.zip`. If it is not found, it checks `/content/archive.zip`. If neither exists, it prompts you to upload `archive.zip` directly in Colab.


In [ ]:
def resolve_archive_path():
    if DRIVE_ARCHIVE_PATH.exists():
        print(f'Using archive from Google Drive: {DRIVE_ARCHIVE_PATH}')
        return DRIVE_ARCHIVE_PATH

    if LOCAL_ARCHIVE_PATH.exists():
        print(f'Using local archive: {LOCAL_ARCHIVE_PATH}')
        return LOCAL_ARCHIVE_PATH

    print('archive.zip was not found in Drive or /content. Please upload archive.zip now.')
    uploaded = files.upload()

    if ARCHIVE_NAME not in uploaded:
        raise FileNotFoundError(f'Expected an uploaded file named {ARCHIVE_NAME}. Uploaded: {list(uploaded)}')

    return LOCAL_ARCHIVE_PATH


archive_path = resolve_archive_path()

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive_path, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

wav_paths = sorted(EXTRACT_DIR.rglob('*.wav'))
print(f'Extracted to: {EXTRACT_DIR}')
print(f'Found {len(wav_paths)} WAV files recursively.')

if not wav_paths:
    raise FileNotFoundError('No .wav files were found after unzipping archive.zip.')


## Build Manifest

The manifest records file paths, Kaggle filename metadata, original sample rates, and original durations. Unknown labels or malformed filenames are skipped and logged.


In [ ]:
def parse_filename(path: Path):
    """Parse Kaggle filename convention: C_NNNNN_BB_SS_OOOOO_RXX.wav."""
    tokens = path.stem.split('_')

    if len(tokens) != 6:
        raise ValueError(f'expected 6 underscore-separated tokens, found {len(tokens)}')

    context_label, cat_id, breed, sex, owner_id, recording_token = tokens
    context_label = context_label.upper()
    human_label = LABEL_FROM_RAW.get(context_label)

    if human_label is None:
        raise ValueError(f'unknown context label: {context_label}')

    if len(recording_token) < 2:
        raise ValueError(f'invalid recording token: {recording_token}')

    return {
        'context_label': context_label,
        'human_label': human_label,
        'cat_id': cat_id.upper(),
        'breed': breed.upper(),
        'sex': sex.upper(),
        'owner_id': owner_id.upper(),
        'recording_session': recording_token[0],
        'vocalization_counter': recording_token[1:],
    }


manifest_rows = []
skipped_rows = []

for wav_path in wav_paths:
    try:
        parsed = parse_filename(wav_path)
    except ValueError as exc:
        skipped_rows.append({
            'filepath': str(wav_path),
            'filename': wav_path.name,
            'reason': str(exc),
        })
        continue

    try:
        info = sf.info(str(wav_path))
        sample_rate = int(info.samplerate)
        duration_seconds = float(info.frames / info.samplerate)
    except Exception as exc:
        skipped_rows.append({
            'filepath': str(wav_path),
            'filename': wav_path.name,
            'reason': f'audio metadata failed: {exc}',
        })
        continue

    manifest_rows.append({
        'filepath': str(wav_path),
        'filename': wav_path.name,
        **parsed,
        'sample_rate': sample_rate,
        'duration_seconds': duration_seconds,
    })

manifest_df = pd.DataFrame(manifest_rows)
skipped_df = pd.DataFrame(skipped_rows)

print(f'Usable labeled files: {len(manifest_df)}')
print(f'Skipped files: {len(skipped_df)}')

if not skipped_df.empty:
    display(skipped_df)

if manifest_df.empty:
    raise ValueError('No usable audio files remained after manifest creation.')

display(manifest_df.head())


## Dataset Validation Report

Accuracy depends heavily on label quality, recording consistency, and the number of samples per class. A small or imbalanced dataset can produce high-looking validation scores that do not generalize to new cats.


In [ ]:
print('Total files:', len(manifest_df))

label_counts = manifest_df['human_label'].value_counts().reindex(LABEL_ORDER, fill_value=0)
dataset_validation_report = label_counts.rename_axis('label').reset_index(name='file_count')
dataset_validation_report['percentage'] = (dataset_validation_report['file_count'] / len(manifest_df) * 100).round(2)

print('\nFiles per prediction label:')
display(dataset_validation_report)

if (label_counts == 0).any():
    missing_labels = label_counts[label_counts == 0].index.tolist()
    print(f'WARNING: Missing labels in dataset: {missing_labels}. The model cannot learn absent classes.')
else:
    imbalance_ratio = float(label_counts.max() / label_counts.min())
    print(f'Class imbalance ratio, largest/smallest: {imbalance_ratio:.2f}x')
    if imbalance_ratio >= 2.0:
        print('WARNING: Class imbalance detected. Class weights will be used, but more balanced data is recommended.')
    else:
        print('Class balance looks reasonable for a first baseline.')

print('\nCount per cat:')
display(manifest_df['cat_id'].value_counts().rename_axis('cat_id').reset_index(name='count'))

duration_stats = manifest_df['duration_seconds'].agg(['min', 'max', 'mean', 'median']).to_frame(name='duration_seconds')
print('\nDuration stats:')
display(duration_stats)

print('\nOriginal sample rates:')
display(manifest_df['sample_rate'].value_counts().rename_axis('sample_rate').reset_index(name='count'))


## Grouped Train / Validation / Test Split

The split is grouped by `cat_id`, so files from the same cat do not appear in multiple splits. This helps avoid data leakage from cat-specific vocal patterns.


In [ ]:
def split_has_all_labels(frame: pd.DataFrame):
    return set(frame['human_label']) == set(LABEL_ORDER)


def grouped_train_val_test_split(df: pd.DataFrame, test_size=0.2, val_size_of_remaining=0.25, max_attempts=100):
    best = None

    for attempt in range(max_attempts):
        random_state = RANDOM_STATE + attempt
        test_splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_val_idx, test_idx = next(test_splitter.split(df, groups=df['cat_id']))

        train_val_df = df.iloc[train_val_idx].copy()
        test_df = df.iloc[test_idx].copy()

        val_splitter = GroupShuffleSplit(n_splits=1, test_size=val_size_of_remaining, random_state=random_state)
        train_rel_idx, val_rel_idx = next(val_splitter.split(train_val_df, groups=train_val_df['cat_id']))

        train_df = train_val_df.iloc[train_rel_idx].copy()
        val_df = train_val_df.iloc[val_rel_idx].copy()

        best = train_df, val_df, test_df

        if all(split_has_all_labels(frame) for frame in best):
            print(f'Found grouped split with all labels represented after {attempt + 1} attempt(s).')
            return best

    print('Warning: Could not find a grouped split where every split has every label. Using the last attempted grouped split.')
    return best


if manifest_df['cat_id'].nunique() < 3:
    raise ValueError('At least 3 unique cat IDs are needed for grouped train/validation/test splitting.')

train_files_df, val_files_df, test_files_df = grouped_train_val_test_split(manifest_df)

manifest_df['split'] = 'unassigned'
manifest_df.loc[train_files_df.index, 'split'] = 'train'
manifest_df.loc[val_files_df.index, 'split'] = 'validation'
manifest_df.loc[test_files_df.index, 'split'] = 'test'

for split_name, split_df in [('train', train_files_df), ('validation', val_files_df), ('test', test_files_df)]:
    print(f'\n{split_name.upper()} files: {len(split_df)}')
    print('Cats:', sorted(split_df['cat_id'].unique()))
    display(split_df['human_label'].value_counts().reindex(LABEL_ORDER, fill_value=0).rename_axis('label').reset_index(name='count'))

missing_train_labels = set(LABEL_ORDER) - set(train_files_df['human_label'])
if missing_train_labels:
    raise ValueError(f'Training split is missing labels: {sorted(missing_train_labels)}. Add more cats/data or adjust the split settings.')

cat_split_counts = manifest_df.groupby('cat_id')['split'].nunique()
if (cat_split_counts > 1).any():
    raise RuntimeError('Data leakage detected: at least one cat_id appears in multiple splits.')

print('\nGrouped split verified: every cat_id appears in only one split.')


## Load Audio and Create 3-Second Windows

The original files are 8000 Hz, but this notebook uses `librosa.load(path, sr=16000, mono=True)` to resample to 16 kHz because YAMNet expects 16 kHz audio.

Windowing rules:

- Shorter than 3 seconds: pad with zeros.
- Exactly 3 seconds: keep as-is.
- Longer than 3 seconds: split into 3-second chunks with 50% overlap. A final tail chunk is added when needed so the end of the audio is represented.


In [ ]:
def make_three_second_windows(waveform: np.ndarray):
    if len(waveform) <= TARGET_SAMPLES:
        return [np.pad(waveform, (0, TARGET_SAMPLES - len(waveform)), mode='constant').astype(np.float32)]

    starts = list(range(0, len(waveform) - TARGET_SAMPLES + 1, HOP_SAMPLES))
    final_start = len(waveform) - TARGET_SAMPLES

    if starts[-1] != final_start:
        starts.append(final_start)

    return [waveform[start:start + TARGET_SAMPLES].astype(np.float32) for start in starts]


def load_audio_windows_for_split(split_df: pd.DataFrame, split_name: str):
    rows = []
    windows = []
    labels = []

    for _, row in split_df.iterrows():
        waveform, _ = librosa.load(row['filepath'], sr=TARGET_SR, mono=True)
        chunks = make_three_second_windows(waveform)

        for chunk_index, chunk in enumerate(chunks):
            windows.append(chunk)
            labels.append(LABEL_TO_INDEX[row['human_label']])
            rows.append({
                'filename': row['filename'],
                'cat_id': row['cat_id'],
                'human_label': row['human_label'],
                'split': split_name,
                'chunk_index': chunk_index,
                'chunk_samples': len(chunk),
            })

    return np.stack(windows).astype(np.float32), np.array(labels, dtype=np.int64), pd.DataFrame(rows)


train_waveforms, y_train, train_chunks_df = load_audio_windows_for_split(train_files_df, 'train')
val_waveforms, y_val, val_chunks_df = load_audio_windows_for_split(val_files_df, 'validation')
test_waveforms, y_test, test_chunks_df = load_audio_windows_for_split(test_files_df, 'test')

chunks_df = pd.concat([train_chunks_df, val_chunks_df, test_chunks_df], ignore_index=True)

print('Train waveform chunks:', train_waveforms.shape)
print('Validation waveform chunks:', val_waveforms.shape)
print('Test waveform chunks:', test_waveforms.shape)

print('\nChunk counts by split and label:')
display(pd.crosstab(chunks_df['split'], chunks_df['human_label']).reindex(columns=LABEL_ORDER, fill_value=0))


def add_random_noise(waveform: np.ndarray, rng: np.random.Generator):
    rms = float(np.sqrt(np.mean(np.square(waveform))))
    noise_scale = max(rms * AUGMENT_NOISE_LEVEL, 1e-4)
    noise = rng.normal(0.0, noise_scale, size=waveform.shape)
    return (waveform + noise).astype(np.float32)


def add_time_shift(waveform: np.ndarray, rng: np.random.Generator):
    max_shift_samples = int(AUGMENT_MAX_SHIFT_SECONDS * TARGET_SR)
    shift = int(rng.integers(-max_shift_samples, max_shift_samples + 1))
    return np.roll(waveform, shift).astype(np.float32)


def add_pitch_shift(waveform: np.ndarray, rng: np.random.Generator):
    direction = float(rng.choice([-1.0, 1.0]))
    shifted = librosa.effects.pitch_shift(
        y=waveform,
        sr=TARGET_SR,
        n_steps=direction * AUGMENT_PITCH_STEPS,
    )
    return shifted.astype(np.float32)


def augment_training_set(waveforms: np.ndarray, labels: np.ndarray, chunk_rows: pd.DataFrame):
    if not ENABLE_AUDIO_AUGMENTATION:
        print('Audio augmentation disabled. Training on original chunks only.')
        return waveforms, labels, chunk_rows

    rng = np.random.default_rng(RANDOM_STATE)
    augmented_waveforms = list(waveforms)
    augmented_labels = list(labels)
    augmented_rows = []

    augmentation_fns = []
    if AUGMENT_RANDOM_NOISE:
        augmentation_fns.append(('noise', add_random_noise))
    if AUGMENT_TIME_SHIFT:
        augmentation_fns.append(('time_shift', add_time_shift))
    if AUGMENT_PITCH_SHIFT:
        augmentation_fns.append(('pitch_shift', add_pitch_shift))

    for index, waveform in enumerate(waveforms):
        source_row = chunk_rows.iloc[index].to_dict()
        for augmentation_name, augmentation_fn in augmentation_fns:
            augmented_waveforms.append(augmentation_fn(waveform, rng))
            augmented_labels.append(int(labels[index]))
            augmented_row = dict(source_row)
            augmented_row['augmentation'] = augmentation_name
            augmented_rows.append(augmented_row)

    original_rows = chunk_rows.copy()
    original_rows['augmentation'] = 'original'
    augmented_rows_df = pd.concat(
        [original_rows, pd.DataFrame(augmented_rows)],
        ignore_index=True,
    )

    augmented_waveforms = np.stack(augmented_waveforms).astype(np.float32)
    augmented_labels = np.array(augmented_labels, dtype=np.int64)

    print(f'Audio augmentation enabled. Training chunks: {len(waveforms)} -> {len(augmented_waveforms)}')
    display(augmented_rows_df['augmentation'].value_counts().rename_axis('augmentation').reset_index(name='count'))

    return augmented_waveforms, augmented_labels, augmented_rows_df


train_waveforms, y_train, train_chunks_df = augment_training_set(train_waveforms, y_train, train_chunks_df)

print('\nTraining chunk counts after optional augmentation:')
display(train_chunks_df['human_label'].value_counts().reindex(LABEL_ORDER, fill_value=0).rename_axis('label').reset_index(name='chunk_count'))


## Extract YAMNet Embeddings

YAMNet is loaded from TensorFlow Hub. Each 3-second waveform chunk is converted to YAMNet embeddings, then embeddings are averaged over time to produce one feature vector per chunk.


In [ ]:
YAMNET_HANDLE = 'https://tfhub.dev/google/yamnet/1'
yamnet_model = hub.load(YAMNET_HANDLE)
print('Loaded YAMNet from TensorFlow Hub.')


def waveform_to_mean_embedding(waveform: np.ndarray):
    scores, embeddings, spectrogram = yamnet_model(waveform)
    del scores, spectrogram
    return tf.reduce_mean(embeddings, axis=0).numpy()


def embed_waveforms(waveforms: np.ndarray, split_name: str):
    embedding_rows = []

    for index, waveform in enumerate(waveforms):
        embedding_rows.append(waveform_to_mean_embedding(waveform))

        if (index + 1) % 50 == 0 or (index + 1) == len(waveforms):
            print(f'{split_name}: embedded {index + 1}/{len(waveforms)} chunks')

    return np.stack(embedding_rows).astype(np.float32)


X_train = embed_waveforms(train_waveforms, 'train')
X_val = embed_waveforms(val_waveforms, 'validation')
X_test = embed_waveforms(test_waveforms, 'test')

print('Train embeddings:', X_train.shape)
print('Validation embeddings:', X_val.shape)
print('Test embeddings:', X_test.shape)


## Class Weights

The dataset is imbalanced, with `I` usually having the most samples and `F` the fewest. Class weights are computed from the training chunks only.


In [ ]:
class_weight_values = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train,
)
class_weights = {index: float(weight) for index, weight in enumerate(class_weight_values)}

print('Class weights:')
for index, weight in class_weights.items():
    print(f'  {index} ({INDEX_TO_LABEL[str(index)]}): {weight:.4f}')


## Train Classifier

Classifier architecture:

- Dense 256 ReLU
- Dropout 0.4
- Dense 3 Softmax


In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
    ]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=16,
    callbacks=[early_stopping],
    class_weight=class_weights,
)


## Evaluate Model

This section reports training, validation, and test accuracy, plus a test confusion matrix and classification report.


In [ ]:
train_loss, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f'Training accuracy: {train_accuracy:.4f}')
print(f'Validation accuracy: {val_accuracy:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')

y_test_pred_proba = model.predict(X_test)
y_test_pred = np.argmax(y_test_pred_proba, axis=1)

print('\nTest classification report:')
print(classification_report(
    y_test,
    y_test_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=LABEL_ORDER,
    zero_division=0,
))

cm = confusion_matrix(y_test, y_test_pred, labels=np.arange(NUM_CLASSES))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABEL_ORDER).plot(
    cmap='Blues',
    values_format='d',
    ax=ax,
    colorbar=False,
)
ax.set_title('Test Confusion Matrix')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df.index + 1, history_df['loss'], label='Train loss')
axes[0].plot(history_df.index + 1, history_df['val_loss'], label='Validation loss')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history_df.index + 1, history_df['accuracy'], label='Train accuracy')
axes[1].plot(history_df.index + 1, history_df['val_accuracy'], label='Validation accuracy')
axes[1].set_title('Training Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
fig.savefig(HISTORY_PLOT_OUTPUT_PATH, dpi=160, bbox_inches='tight')
plt.show()

print(f'Saved training history plot: {HISTORY_PLOT_OUTPUT_PATH.resolve()}')


## Save Artifacts

This saves:

- `cat_model.keras`
- `cat_model_<timestamp>.keras`
- `label_map.json`
- `manifest.csv`
- `training_history_<timestamp>.png`


In [ ]:
model.save(MODEL_OUTPUT_PATH)
model.save(TIMESTAMPED_MODEL_OUTPUT_PATH)

with LABEL_MAP_OUTPUT_PATH.open('w', encoding='utf-8') as file:
    json.dump(INDEX_TO_LABEL, file, indent=2)

manifest_df[MANIFEST_COLUMNS].to_csv(MANIFEST_OUTPUT_PATH, index=False)

print(f'Saved model: {MODEL_OUTPUT_PATH.resolve()}')
print(f'Saved timestamped model version: {TIMESTAMPED_MODEL_OUTPUT_PATH.resolve()}')
print(f'Saved label map: {LABEL_MAP_OUTPUT_PATH.resolve()}')
print(f'Saved manifest: {MANIFEST_OUTPUT_PATH.resolve()}')
print(f'Saved training history plot: {HISTORY_PLOT_OUTPUT_PATH.resolve()}')
print('\nLabel map:')
print(json.dumps(INDEX_TO_LABEL, indent=2))


## Download Artifacts

Run this final cell to download the trained classifier, timestamped model version, label map, manifest, and training history plot from Colab.


In [ ]:
files.download(str(MODEL_OUTPUT_PATH))
files.download(str(TIMESTAMPED_MODEL_OUTPUT_PATH))
files.download(str(LABEL_MAP_OUTPUT_PATH))
files.download(str(MANIFEST_OUTPUT_PATH))
files.download(str(HISTORY_PLOT_OUTPUT_PATH))
